In [1]:
import os, random
import pandas as pd

from torch.utils.data import Subset
import numpy as np

from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from PIL import ImageFile, UnidentifiedImageError
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

from torchvision.models import resnet50

INPUT_DIR = "/kaggle/input/datasets/karvysingh/wikiart"
IMAGES_ROOT = os.path.join(INPUT_DIR, "wikiart_clean", "wikiart")
TRAIN_CSV = os.path.join("/kaggle/input/datasets/karvysingh/trainfull", "style_train_clean.csv")
VAL_CSV   = os.path.join(INPUT_DIR, "style_val_lite_clean.csv")


In [2]:
def read_wikiart_csv(path):
    df = pd.read_csv(path)

    # headerless got misread as header (common)
    if len(df.columns) == 2:
        c0, c1 = df.columns[0], df.columns[1]
        if ("/" in str(c0)) and str(c1).strip().isdigit():
            df = pd.read_csv(path, header=None, names=["relpath", "label"])
            df["label"] = df["label"].astype(int)
            return df

    # normal header (strip spaces)
    df.columns = [c.strip() for c in df.columns]

    # guess columns
    path_col = df.columns[0]
    label_col = df.columns[1]
    for c in df.columns:
        lc = c.lower()
        if lc in ["pictures", "picture", "path", "file", "filename", "image"]:
            path_col = c
        if lc in ["class", "label", "target"]:
            label_col = c

    df = df[[path_col, label_col]].rename(columns={path_col:"relpath", label_col:"label"})
    df["label"] = df["label"].astype(int)
    return df


def filter_bad_images(df, images_root):
    good_rows = []
    bad = 0

    for relpath, label in zip(df["relpath"].astype(str).values, df["label"].values):
        fullpath = os.path.join(images_root, relpath)
        try:
            with Image.open(fullpath) as im:
                im.verify()  # verifies file integrity without decoding full image
            good_rows.append((relpath, int(label)))
        except (OSError, UnidentifiedImageError):
            bad += 1

    out = pd.DataFrame(good_rows, columns=["relpath", "label"])
    print(f"Filtered bad images: {bad} removed, {len(out)} remaining")
    return out

In [3]:
class WikiArtDataset(Dataset):
    def __init__(self, df, images_root, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.images_root = images_root
        self.transform = transform

        self.df["fullpath"] = self.df["relpath"].astype(str).apply(lambda p: os.path.join(images_root, p))
        # (you already verified 0 missing, so no filtering here)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["fullpath"]
        label = int(row["label"])
    
        img = Image.open(img_path)
        img = img.convert("RGB") 
        if self.transform:
            img = self.transform(img)
        return img, label

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_df = read_wikiart_csv(TRAIN_CSV)
SUBSET_N = 10000  
train_df = train_df.sample(n=min(SUBSET_N, len(train_df)), random_state=42).reset_index(drop=True)
train_df = filter_bad_images(train_df, IMAGES_ROOT)

val_df   = read_wikiart_csv(VAL_CSV)

# num classes from max label + 1 (safer than nunique)
num_classes = int(max(train_df["label"].max(), val_df["label"].max())) + 1
print("num_classes:", num_classes)
print("train/val sizes:", len(train_df), len(val_df))

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_dataset = WikiArtDataset(train_df, IMAGES_ROOT, transform=train_transform)
val_dataset   = WikiArtDataset(val_df, IMAGES_ROOT, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

model = resnet50(weights='DEFAULT')

for p in model.parameters():
    p.requires_grad = False

for p in model.layer2.parameters():
    p.requires_grad = True

for p in model.layer3.parameters():
    p.requires_grad = True

for p in model.layer4.parameters():
    p.requires_grad = True

model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

params = [p for p in model.parameters() if p.requires_grad]

optimizer = optim.Adam(params, lr=3e-4)

use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler('cuda',enabled=use_amp)

# ----------------------
# Metrics
# ----------------------
def topk_acc(logits, y, k=5):
    k = min(k, logits.size(1))
    _, topk = logits.topk(k, dim=1)
    return (topk == y.unsqueeze(1)).any(dim=1).float().mean().item()

# ----------------------
# Train
# ----------------------

def mixup_data(x, y, alpha=0.2):
    if alpha <= 0:
        return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

FAST_TRAIN_STEPS = 30   # number of train batches per epoch (fast iteration)

epochs = 8
steps_per_epoch = FAST_TRAIN_STEPS

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.3)

for epoch in range(1, epochs+1):
    model.train()
    loss_sum = 0.0
    steps = 0

    for i, (x, y) in enumerate(train_loader):
        if i >= FAST_TRAIN_STEPS:
            break

        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        x_mix, y_a, y_b, lam = mixup_data(x, y, alpha=0.2)
        
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            logits = model(x_mix)
            loss = mixup_criterion(criterion, logits, y_a, y_b, lam)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


        loss_sum += loss.item()
        steps += 1
        

    scheduler.step()
    
    # Validate (fast)
    model.eval()
    correct1 = 0
    total = 0
    top5_sum = 0.0
    batches = 0

    with torch.no_grad():
        for x,y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
    
            with torch.amp.autocast('cuda',enabled=use_amp):
                logits = model(x)
            pred = logits.argmax(dim=1)

            total += y.size(0)
            correct1 += (pred == y).sum().item()

            top5_sum += topk_acc(logits, y, k=5)
            batches += 1

    val_top1 = 100.0 * correct1 / max(1, total)
    val_top5 = 100.0 * (top5_sum / max(1, batches))

    print(f"Epoch {epoch:02d}/{epochs} | train_loss {loss_sum/max(1,steps):.4f} | val_top1 {val_top1:.2f}% | val_top5 {val_top5:.2f}%")

Device: cuda
Filtered bad images: 0 removed, 10000 remaining
num_classes: 27
train/val sizes: 10000 10178
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 191MB/s] 


Epoch 01/8 | train_loss 2.6378 | val_top1 25.06% | val_top5 64.53%
Epoch 02/8 | train_loss 1.9763 | val_top1 35.95% | val_top5 77.80%
Epoch 03/8 | train_loss 1.7825 | val_top1 39.08% | val_top5 81.13%
Epoch 04/8 | train_loss 1.6770 | val_top1 42.29% | val_top5 83.37%
Epoch 05/8 | train_loss 1.6174 | val_top1 45.77% | val_top5 86.23%
Epoch 06/8 | train_loss 1.4723 | val_top1 47.86% | val_top5 86.47%
Epoch 07/8 | train_loss 1.3558 | val_top1 49.13% | val_top5 87.39%
Epoch 08/8 | train_loss 1.3278 | val_top1 47.49% | val_top5 86.30%
